In [1]:
import pandas as pd
from google.colab import files
import requests
import json



In [3]:
# read the excel file
# You have to upload by yourself
df = pd.read_excel('/content/data kinase.xlsx')

# extract the first and second columns and remove duplicates
substrate_values = df.iloc[:, 0].dropna().unique().tolist()
kinase_values = df.iloc[:, 1].dropna().unique().tolist()



In [4]:
# Re-import necessary modules due to environment reset
import aiohttp
import asyncio
import json
from tqdm.asyncio import tqdm as tqdm_asyncio

# Async function to fetch UniProt data with retry mechanism
async def fetch_protein_info_with_retry(session, uniprot_id, max_retries=5):
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
    for attempt in range(max_retries):
        try:
            async with session.get(url, timeout=10) as response:
                if response.status == 200:
                    data = await response.json()

                    # Extract Function comment
                    comments = data.get("comments", [])
                    function_texts = []
                    for comment in comments:
                        if comment.get("commentType") == "FUNCTION":
                            texts_list = comment.get("texts", [])
                            for text_item in texts_list:
                                if "value" in text_item:
                                    function_texts.append(text_item["value"])
                    function_info = "\n".join(function_texts) if function_texts else None

                    # Extract Protein name
                    protein_name = data.get("proteinDescription", {}).get("recommendedName", {}).get("fullName", {}).get("value", None)

                    # Extract Gene name
                    genes = data.get("genes", [])
                    gene_name = genes[0].get("geneName", {}).get("value", None) if genes else None

                    combined_info = {
                        "Protein_name": protein_name,
                        "Gene": gene_name,
                        "Function": function_info
                    }
                    return uniprot_id, combined_info

                else:
                    err_msg = f"HTTP {response.status}"
        except Exception as e:
            err_msg = f"Exception: {e}"

        print(f"Warning: Failed to fetch {uniprot_id} (attempt {attempt+1}/{max_retries}): {err_msg}")
        await asyncio.sleep(2 ** attempt)  # Exponential backoff

    return uniprot_id, {"Function": None, "Protein_name": None, "Gene": None}

# Main async function to process all UniProt IDs
async def fetch_all_protein_infos(uniprot_ids, concurrency=20):
    connector = aiohttp.TCPConnector(limit_per_host=concurrency)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [fetch_protein_info_with_retry(session, uid) for uid in uniprot_ids]
        results = await tqdm_asyncio.gather(*tasks)
    return dict(results)


In [5]:
# --- Running part ---
# Assuming uniprot_ids is a list of UniProt IDs, e.g.:
# uniprot_ids = ['Q2EF73', 'P0DTD1', 'Q6Q788', 'P0A9R6']

# Example: convert the column1_values to a list if it's in another format

# Fetch protein information for all UniProt IDs
protein_info_map = await fetch_all_protein_infos(substrate_values)


# Optional: You can filter out invalid information
valid_protein_info = [
    info for info in protein_info_map.values()
    if not all(value is None for value in info.values())
]
# Optionally save to a JSON file
output_json_filename = "protein_info_data.json"
with open(output_json_filename, "w", encoding="utf-8") as f:
    json.dump(protein_info_map, f, ensure_ascii=False, indent=2)

# Corrected the print statement by adding the closing parenthesis
print(f"Protein information saved to '{output_json_filename}'")

100%|██████████| 914/914 [00:08<00:00, 105.03it/s]

Protein information saved to 'protein_info_data.json'


In [7]:
# upload this file by yourself, I make this file manually , lots of gene can not search on the database
df_kinase = pd.read_excel('/content/only kinase.xlsx', header=None, skiprows=1)

kinase_col1 = df_kinase.iloc[:, 0].dropna().tolist()
kinase_col2 = df_kinase.iloc[:, 1].dropna().tolist()

kinase_lookup = dict(zip(kinase_col2, kinase_col1))


In [8]:
# Fetch protein information for all kinases IDs
protein_info_map = await fetch_all_protein_infos(kinase_col2)

valid_protein_info = [
    info for info in protein_info_map.values()
    if not all(value is None for value in info.values())
]
# Optionally save to a JSON file
output_json_filename = "kinase_info_data.json"
with open(output_json_filename, "w", encoding="utf-8") as f:
    json.dump(protein_info_map, f, ensure_ascii=False, indent=2)

# Corrected the print statement by adding the closing parenthesis
print(f"Protein information saved to '{output_json_filename}'")

100%|██████████| 193/193 [00:02<00:00, 76.28it/s] 

Protein information saved to 'kinase_info_data.json'


In [9]:

# Load the kinase_info_data.json file
with open("kinase_info_data.json", "r", encoding="utf-8") as f:
    kinase_data = json.load(f)

# Create a new dictionary with updated keys
updated_kinase_data = {}
for uniprot_id, info in kinase_data.items():
    # Check if the UniProt ID exists in the kinase_lookup dictionary
    if uniprot_id in kinase_lookup:
        protein_name = kinase_lookup[uniprot_id]
        updated_kinase_data[protein_name] = info
    else:
        # If not found in the lookup, keep the original UniProt ID as the key
        updated_kinase_data[uniprot_id] = info


# Optionally save the updated data to a new JSON file
updated_output_json_filename = "kinase_info_data_updated_keys.json"
with open(updated_output_json_filename, "w", encoding="utf-8") as f:
    json.dump(updated_kinase_data, f, ensure_ascii=False, indent=2)

print(f"Updated kinase information saved to '{updated_output_json_filename}'")



Updated kinase information saved to 'kinase_info_data_updated_keys.json'


In [10]:

# Load the protein_info_data.json file
with open("protein_info_data.json", "r", encoding="utf-8") as f:
    protein_data = json.load(f)

with open("kinase_info_data_updated_keys.json", "r", encoding="utf-8") as f:
    kinase_data = json.load(f)

# Combine the two dictionaries
# Note: If there are overlapping keys, the value from protein_data will overwrite the value from updated_kinase_data
merged_data = {**kinase_data, **protein_data}

# Optionally save the merged data to a new JSON file
merged_output_json_filename = "all_database_info.json"
with open(merged_output_json_filename, "w", encoding="utf-8") as f:
    json.dump(merged_data, f, ensure_ascii=False, indent=2)

print(f"Merged data saved to '{merged_output_json_filename}'")


Merged data saved to 'all_database_info.json'


In [ ]:
告诉我第一个文件有多少个键